# 0. les biblios

In [ ]:
import pandas as pd

import re

import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np

# 1. Charger monitoring_device_operations.csv

In [ ]:
# Charger les données

df = pd.read_csv("monitoring_device_operations.csv")

In [ ]:
# 1. Dimensions et types de colonnes

print("Dimensions :", df.shape)

print("\nTypes de colonnes :")

df.info()

In [ ]:
# 2. Échantillon des données

print("\nPremières lignes :")

display(df.head())

- 4994 lignes et 11 colonnes.

- Une ligne = une opération mensuelle d'un appareil de monitoring sur un site (`site_code` + `device_reference` + `operation_start`).

- Toutes les colonnes sont en `object`, y compris les mesures et les dates : même signal qu'avant, des formats sont mélangés.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Nombre de lignes strictement dupliquées :", df.duplicated().sum())

display(df[df.duplicated(keep=False)].sort_values('device_operation_id'))

### b- Doublons sur l'identifiant métier



`device_operation_id` devrait être un identifiant unique par opération.

In [ ]:
print("device_operation_id dupliqués :", df['device_operation_id'].duplicated().sum())

display(df[df['device_operation_id'].duplicated(keep=False)].sort_values('device_operation_id'))



# Vérification croisée avec la clé (site, appareil, début d'opération)

cle = ['site_code', 'device_reference', 'operation_start']

print("Couples (site, appareil, début) dupliqués :", df.duplicated(subset=cle).sum())

- Solution

In [ ]:
# On supprime les doublons stricts puis les doublons d'identifiant (on garde la 1ère occurrence)

df = df.drop_duplicates(keep='first')

df = df.drop_duplicates(subset='device_operation_id', keep='first')



# Vérification

print("Nombre de lignes après suppression :", df.shape[0])

print("Doublons stricts restants :", df.duplicated().sum())

print("Doublons d'identifiant restants :", df['device_operation_id'].duplicated().sum())

### c- Doublons de colonnes

In [ ]:
noms_doublons = df.columns[df.columns.duplicated()]

print("Noms de colonnes en double :", noms_doublons.tolist())



contenu_doublons = df.columns[df.T.duplicated()]

print("Colonnes avec un contenu dupliqué :", contenu_doublons.tolist())

In [ ]:
# site_code et device_reference se correspondent-ils parfaitement ?

print("Appareils par site :", df.groupby('site_code')['device_reference'].nunique().value_counts())

print("Sites par appareil :", df.groupby('device_reference')['site_code'].nunique().value_counts())

→ Correspondance 1-pour-1 entre `site_code` et `device_reference` : chaque site n'a qu'un seul appareil dans

ce jeu de données. On garde les deux colonnes : l'une identifie le lieu, l'autre le matériel — utile pour la

traçabilité si un appareil est un jour remplacé sur le même site.

# 3. Formats incohérents

### Les types de chaque colonne :

In [ ]:
print(df.dtypes)

### - Vérifier le format de device_operation_id et device_reference

In [ ]:
print("Format device_operation_id :", df['device_operation_id'].str.replace(r'\d', '9', regex=True).unique())

print("Format device_reference :", df['device_reference'].str.replace(r'[A-Z0-9]', 'X', regex=True).unique())

print("Manquants :", df['device_operation_id'].isna().sum(), df['device_reference'].isna().sum())

→ Formats homogènes (`DOP-XXXXXXX` et `DEV-XXXXXX-X`). Rien à corriger.

### - Vérifier le format de operation_start / operation_end (dates) <--->

In [ ]:
start_dt = pd.to_datetime(df['operation_start'], errors='coerce')

end_dt = pd.to_datetime(df['operation_end'], errors='coerce')



print("operation_start non convertibles :", start_dt.isna().sum() - df['operation_start'].isna().sum())

print("operation_end non convertibles :", end_dt.isna().sum() - df['operation_end'].isna().sum())



# Exemples de formats qui posent problème

mask_bad = end_dt.isna() & df['operation_end'].notna()

print("\nExemples de operation_end non convertibles :")

display(df.loc[mask_bad, ['device_operation_id', 'operation_start', 'operation_end']].head(10))

**Constat :** `operation_start` est toujours au format ISO, mais `operation_end` mélange plusieurs formats :

`31-May-2025 23:59:59`, `2022/11/30 23:59:59`, `30/04/2019 23:59:59`. C'est exactement le cas "format

variable" décrit dans le cours.

- Solution

In [ ]:
# operation_start est toujours au format ISO : on le précise pour éviter que le mode "mixed"

# ne le confonde avec un format day-first (piège classique : 2019-02-01 lu comme le 2 janvier)

df['operation_start'] = pd.to_datetime(df['operation_start'], format='ISO8601', errors='coerce')



# operation_end mélange plusieurs formats (dont certains ambigus type JJ/MM/AAAA) : on laisse

# pandas déduire le format ligne par ligne, avec dayfirst pour lever l'ambiguïté JJ/MM vs MM/JJ

df['operation_end'] = pd.to_datetime(df['operation_end'], format='mixed', errors='coerce', dayfirst=True)



# Vérification

print("operation_start non convertibles :", df['operation_start'].isna().sum())

print("operation_end non convertibles :", df['operation_end'].isna().sum())

print("Lignes où end <= start :", (df['operation_end'] <= df['operation_start']).sum())

print("Types :", df['operation_start'].dtype, df['operation_end'].dtype)

### - Vérifier le format de uptime_pct et acoustic_quality (numériques) <--->

In [ ]:
for col in ['uptime_pct', 'acoustic_quality']:

    s = df[col]

    non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

    print(col, "| non convertibles :", len(non_num), "| exemples :", non_num.unique()[:8])

**Constat :** `uptime_pct` a le même problème de virgule décimale que dans `environmental_monthly.csv`.

- Solution

In [ ]:
df['uptime_pct'] = df['uptime_pct'].astype(str).str.replace(',', '.', regex=False)

df['uptime_pct'] = pd.to_numeric(df['uptime_pct'], errors='coerce')

df['acoustic_quality'] = pd.to_numeric(df['acoustic_quality'], errors='coerce')



# Vérification

print(df[['uptime_pct', 'acoustic_quality']].dtypes)

print(df[['uptime_pct', 'acoustic_quality']].isna().sum())

### - Vérifier le format de maintenance_type (catégorielle) <--->

In [ ]:
print(df['maintenance_type'].value_counts(dropna=False))

**Constat :** 5 valeurs pour 3 catégories réelles (`routine inspection` / `ROUTINE INSPECTION` /

`Routine Inspection`). Même correction que pour `land_use_category` dans le fichier écologique.

- Solution

In [ ]:
df['maintenance_type'] = df['maintenance_type'].str.strip().str.lower()



# Vérification

print(df['maintenance_type'].value_counts())

### - Vérifier le format de record_status (catégorielle)

In [ ]:
print(df['record_status'].value_counts(dropna=False))

- Solution

In [ ]:
df['record_status'] = df['record_status'].str.strip().str.lower()

print(df['record_status'].value_counts())

→ Une seule vraie modalité (`closed`) : cette colonne n'apporte aucune information (variance nulle),

elle sera écartée des analyses de corrélation/visualisation mais conservée dans le fichier exporté.

### - Vérifier le format de maintenance_provider (catégorielle)

In [ ]:
print(df['maintenance_provider'].value_counts(dropna=False))

print("Manquants :", df['maintenance_provider'].isna().sum())

# 4. Valeurs manquantes

### a. Constat du problème

In [ ]:
print("Valeurs manquantes par colonne :")

print(df.isna().sum())



print("\nPourcentage de manquants par colonne :")

print((df.isna().sum() / len(df) * 100).round(2))

- `fault_code` : ~96% de manquants

- `maintenance_provider` : 30 manquants (0.6%)



Deux logiques différentes, comme pour `environmental_monthly.csv`.

### b. fault_code : l'absence de code EST une information



Hypothèse : un code de panne n'est renseigné que lors d'une intervention corrective, pas lors d'une

inspection de routine.

In [ ]:
print(pd.crosstab(df['maintenance_type'], df['fault_code'], dropna=False))

→ Preuve par les données : `fault_code` est rempli à 100% pour `corrective maintenance` et

`sensor replacement`, et jamais pour `routine inspection`. Le vide correspond donc à « aucune panne » et

non à une perte d'information.

In [ ]:
df['fault_code'] = df['fault_code'].fillna('NO_FAULT')



# Vérification

print(df['fault_code'].value_counts())

print("Manquants restants :", df['fault_code'].isna().sum())

### c. maintenance_provider : inférence par appareil



Hypothèse : chaque site/appareil est toujours suivi par le même prestataire de maintenance.

In [ ]:
trous = df[df['maintenance_provider'].isna()]

print("Lignes concernées :", len(trous), "sur", trous['site_code'].nunique(), "sites différents")



sites_a_verifier = trous['site_code'].unique()

for site in sites_a_verifier[:10]:

    prestataires_connus = df[(df['site_code'] == site) & (df['maintenance_provider'].notna())]['maintenance_provider'].unique()

    print(site, "->", prestataires_connus)

→ Résultat : chaque site n'a qu'un seul prestataire dans tout le reste du dataset. Hypothèse confirmée.

- Solution

In [ ]:
df['maintenance_provider'] = df.groupby('site_code')['maintenance_provider'].transform(

    lambda s: s.fillna(s.mode()[0]) if not s.mode().empty else s

)



# Vérification

print("Valeurs manquantes restantes :")

print(df.isna().sum())

# 5. Gestion des variables catégorielles

### Typage des variables (Cours 1)

| Variable | Type | Rôle |
|---|---|---|
| `device_operation_id` | nominale (identifiant) | clé unique d'opération |
| `site_code` | nominale | site de monitoring |
| `device_reference` | nominale (identifiant) | appareil (1-pour-1 avec le site) |
| `operation_start` / `operation_end` | dates | bornes du cycle → transformées en `year`, `month`, `duration_days` |
| `uptime_pct` | numérique continue | % de disponibilité de l'appareil |
| `acoustic_quality` | numérique continue | score de qualité acoustique |
| `duration_days` | numérique discrète (dérivée) | durée du cycle en jours |
| `year` / `month` | numérique discrète (dérivées) | composantes temporelles |
| `maintenance_type` | nominale non ordonnée | type d'intervention |
| `fault_code` | nominale non ordonnée | code de panne (`NO_FAULT` si aucune) |
| `maintenance_provider` | nominale non ordonnée | prestataire |
| `record_status` | nominale (variance nulle) | une seule modalité `closed` → peu informative |


### a. Création de variables temporelles

In [ ]:
df['year'] = df['operation_start'].dt.year

df['month'] = df['operation_start'].dt.month

df['duration_days'] = (df['operation_end'] - df['operation_start']).dt.days



display(df.head())

### b. Variables ordonnées

Aucune variable réellement ordinale : `maintenance_type`, `fault_code` et `maintenance_provider` sont des

catégories sans hiérarchie naturelle. `record_status` n'a qu'une seule modalité.

### c. Encodage One-Hot des variables nominales

`maintenance_type`, `fault_code` et `maintenance_provider` sont des variables **nominales** (aucun ordre entre les modalités) : on les encode en One-Hot avec `pd.get_dummies`.

- `record_status` est écartée (une seule modalité) : elle n'apporte aucune information.
- `df_stats` reste **non encodé** (export lisible) ; l'encodage est stocké dans `df_encoded`, exporté à part.


On encode **maintenant** sur ce fichier (comme pour `environmental_monthly`), afin de disposer d'une version prête pour le clustering / la corrélation avec les dummies. Un ré-encodage global après merge restera possible si les modalités divergent d'un fichier à l'autre.


In [ ]:
df_stats = df.copy()



print(df_stats.dtypes)

display(df_stats.head())

In [ ]:
cols_nominales = ['maintenance_type', 'fault_code', 'maintenance_provider']

df_encoded = pd.get_dummies(df_stats, columns=cols_nominales, drop_first=False, dtype=int)

# record_status : variance nulle → on la retire de la version encodée
if 'record_status' in df_encoded.columns:
    df_encoded = df_encoded.drop(columns=['record_status'])

cols_maint = [c for c in df_encoded.columns if c.startswith('maintenance_type_')]
cols_fault = [c for c in df_encoded.columns if c.startswith('fault_code_')]
cols_provider = [c for c in df_encoded.columns if c.startswith('maintenance_provider_')]

print('Avant :', df_stats.shape, '-> Après :', df_encoded.shape)
print('Valeurs manquantes :', df_encoded.isna().sum().sum())
for nom, cols in [('maintenance_type', cols_maint), ('fault_code', cols_fault), ('maintenance_provider', cols_provider)]:
    print(f"{nom} : somme des dummies == 1 sur toutes les lignes ->", bool((df_encoded[cols].sum(axis=1) == 1).all()))

display(df_encoded[cols_maint + cols_fault + cols_provider].head())


# 6. Voir les outliers (colonnes numériques)

In [ ]:
cols_num = ['uptime_pct', 'acoustic_quality', 'duration_days']

display(df_stats[cols_num].describe().T)

In [ ]:
plt.figure(figsize=(14, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.boxplot(y=df_stats[col], color="skyblue")

    plt.title(col)

plt.tight_layout()

plt.show()

In [ ]:
for col in cols_num:

    q1, q3 = df_stats[col].quantile([0.25, 0.75])

    iqr = q3 - q1

    bas, haut = q1 - 1.5 * iqr, q3 + 1.5 * iqr

    n = ((df_stats[col] < bas) | (df_stats[col] > haut)).sum()

    print(f"{col:20s} min={df_stats[col].min():7.1f} max={df_stats[col].max():7.1f} -> {n} points hors moustaches")

### - Justification du traitement des outliers

- Aucune valeur impossible : pourcentages entre 0 et 100, pas de négatif.

- Les `uptime_pct`/`acoustic_quality` bas sont directement expliqués par le type de maintenance :

In [ ]:
print(df_stats.groupby('maintenance_type')[['uptime_pct', 'acoustic_quality']].mean().round(1))

→ Les mois de `corrective maintenance` et surtout `sensor replacement` ont un uptime/qualité acoustique

nettement plus faibles que les inspections de routine : ce sont des réalités opérationnelles (l'appareil est

en panne ou hors service pendant la réparation), pas des erreurs de mesure. **On les conserve.**

# 7. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(14, 5))

for i, col in enumerate(cols_num, 1):

    plt.subplot(1, 3, i)

    sns.histplot(df_stats[col], kde=True, bins=30, color='skyblue')

    plt.title(f"Distribution : {col}")

plt.tight_layout()

plt.show()

In [ ]:
print(df_stats[cols_num].skew().round(2))

### - Analyse exploratoire



- `uptime_pct` et `acoustic_quality` sont fortement asymétriques vers la gauche (skew négatif) : la

majorité des opérations sont de bonne qualité, avec une traîne de valeurs basses liée aux interventions

correctives — comportement de type loi de puissance inversée plutôt qu'une loi normale classique.

- `duration_days` est très concentré (presque toutes les opérations durent ~28-31 jours, un cycle mensuel),

donc peu de variance à interpréter ici.

### - Répartition des variables catégorielles

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

sns.countplot(y='maintenance_type', data=df_stats,
              order=df_stats['maintenance_type'].value_counts().index, color='skyblue', ax=axes[0])
axes[0].set_title('Répartition des types de maintenance')

sns.countplot(y='fault_code', data=df_stats,
              order=df_stats['fault_code'].value_counts().index, color='skyblue', ax=axes[1])
axes[1].set_title('Répartition des codes de panne')

sns.countplot(y='maintenance_provider', data=df_stats,
              order=df_stats['maintenance_provider'].value_counts().index, color='skyblue', ax=axes[2])
axes[2].set_title('Répartition des prestataires')

plt.tight_layout()
plt.show()


### - Tableaux croisés (variables catégorielles)

On croise les catégories deux à deux pour voir les associations : quel type de maintenance va avec quel prestataire / quel code de panne.


In [ ]:
print("maintenance_type x fault_code (effectifs)")
display(pd.crosstab(df_stats["maintenance_type"], df_stats["fault_code"]))

print("\nmaintenance_type x maintenance_provider (% en ligne)")
display((pd.crosstab(df_stats["maintenance_type"], df_stats["maintenance_provider"], normalize="index") * 100).round(1))

print("\nfault_code x maintenance_provider (effectifs)")
display(pd.crosstab(df_stats["fault_code"], df_stats["maintenance_provider"]))


### - Lecture des tableaux croisés

- `routine inspection` est presque toujours associée à `NO_FAULT` : pas de panne déclarée lors des contrôles de routine.
- `corrective maintenance` et `sensor replacement` concentrent les vrais codes de panne : c'est cohérent avec le nettoyage (`fault_code` porteur de sens).
- Le croisement avec `maintenance_provider` montre si un prestataire est sur-représenté sur les interventions correctives (piste de qualité de service / charge de panne).


### - Tendances temporelles

Évolution du volume d'opérations et de la qualité (`uptime_pct`, `acoustic_quality`) dans le temps (par année, puis par mois calendaire).


In [ ]:
# Volume d'événements par année et par mois calendaire
par_annee = df_stats.groupby("year").size()
par_mois = df_stats.groupby("month").size()

# Moyennes de qualité dans le temps
qual_annee = df_stats.groupby("year")[["uptime_pct", "acoustic_quality"]].mean()
qual_mois = df_stats.groupby("month")[["uptime_pct", "acoustic_quality"]].mean()

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

par_annee.plot(kind="bar", ax=axes[0, 0], color="steelblue")
axes[0, 0].set_title("Nombre d'opérations par année")
axes[0, 0].set_xlabel("Année")
axes[0, 0].set_ylabel("Effectif")

par_mois.plot(kind="bar", ax=axes[0, 1], color="teal")
axes[0, 1].set_title("Nombre d'opérations par mois calendaire")
axes[0, 1].set_xlabel("Mois")
axes[0, 1].set_ylabel("Effectif")

qual_annee.plot(ax=axes[1, 0], marker="o")
axes[1, 0].set_title("Uptime / qualité acoustique moyens par année")
axes[1, 0].set_xlabel("Année")
axes[1, 0].set_ylabel("Moyenne")
axes[1, 0].legend(loc="best")

qual_mois.plot(ax=axes[1, 1], marker="o")
axes[1, 1].set_title("Uptime / qualité acoustique moyens par mois")
axes[1, 1].set_xlabel("Mois")
axes[1, 1].set_ylabel("Moyenne")
axes[1, 1].legend(loc="best")

plt.tight_layout()
plt.show()

print("Effectifs par année :")
display(par_annee.to_frame("n_operations"))
print("Types de maintenance par année (%):")
display((pd.crosstab(df_stats["year"], df_stats["maintenance_type"], normalize="index") * 100).round(1))


### - Premières irrégularités suspectes (lecture visuelle)

Points visibles à ce stade de l'EDA :

- **Valeurs extrêmes basses** sur `uptime_pct` / `acoustic_quality` (boxplots) : surtout les mois de `corrective maintenance` et `sensor replacement`, pas des erreurs.
- **Modalités rares** : codes de panne hors `NO_FAULT` (~4 % des lignes) et interventions non-routine → queues que les barplots mettent en évidence.
- **Temporel** : possible pic ou creux d'effectifs (année/mois incomplets, ex. 2025 tronquée), ou baisse nette d'uptime / qualité — à croiser avec `bat_surveys` / field reports (période autour d'avril 2024).
- **`duration_days`** très concentré (~28–31 j) : peu d'information, quasi cycle mensuel fixe.

*Ces observations EDA sont reprises et formalisées au Bloc 3 (anomalies statistiques / ML).*


# 8. Corrélations

In [ ]:
cols_corr = ['uptime_pct', 'acoustic_quality', 'duration_days']
matrice_spearman = df_stats[cols_corr].corr(method='spearman')
matrice_pearson = df_stats[cols_corr].corr(method='pearson')

print('Spearman')
display(matrice_spearman.round(2))
print('Pearson')
display(matrice_pearson.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(matrice_pearson, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f',
            linewidths=0.5, ax=axes[0])
axes[0].set_title('Matrice de corrélation de Pearson')

sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt='.2f',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Matrice de corrélation de Spearman')

plt.tight_layout()
plt.show()


In [ ]:
sns.pairplot(df_stats[['uptime_pct', 'acoustic_quality', 'duration_days']],

             kind='scatter', plot_kws={'alpha': 0.3, 'color': 'teal', 's': 15})

plt.suptitle("Nuages de points croisés", y=1.02)

plt.show()

### - Analyse des corrélations



- **`uptime_pct` ↔ `acoustic_quality`** : corrélation positive attendue — un appareil qui tombe hors service

enregistre aussi une moins bonne qualité acoustique pendant la même période.

- **`duration_days`** ne corrèle avec rien : la durée d'un cycle d'opération est quasi fixe (mensuelle), ce

n'est pas un facteur explicatif ici.

- Comme précédemment, **corrélation n'est pas causalité** : le lien réel passe par une variable cachée commune

(le type de maintenance en cours), pas par un effet direct de l'une sur l'autre.

# 9. Export du jeu de données nettoyé

On exporte deux versions, **sans écraser** l'une par l'autre :

- `monitoring_device_operations_clean.csv` : nettoyé, catégories en texte (`df_stats`) ;
- `monitoring_device_operations_clean_encoded.csv` : même contenu avec One-Hot (`df_encoded`).


In [ ]:
from pathlib import Path
Path('cleaned').mkdir(exist_ok=True)

df_stats.to_csv('cleaned/monitoring_device_operations_clean.csv', index=False)
df_encoded.to_csv('cleaned/monitoring_device_operations_clean_encoded.csv', index=False)

print('Clean   :', df_stats.shape, '| manquants :', df_stats.isna().sum().sum())
print('Encodé  :', df_encoded.shape, '| manquants :', df_encoded.isna().sum().sum())


# 10. Synthèse du nettoyage





| Problème identifié | Colonnes concernées | Traitement appliqué |

|---|---|---|

| Doublons stricts (2) et doublons d'identifiant | toutes | suppression, on garde la 1ère occurrence |

| Dates au format mixte (`31-May-2025`, `2022/11/30`...) | `operation_end` | conversion `format='mixed'` en `datetime` |

| Séparateur décimal virgule | `uptime_pct` | remplacement `,` → `.` |

| Casse incohérente (5 valeurs pour 3 catégories) | `maintenance_type` | normalisation en minuscules |

| Casse incohérente (3 valeurs pour 1 catégorie) | `record_status` | normalisation en minuscules |

| Manquants "porteurs de sens" (96%) | `fault_code` | remplacés par la modalité `NO_FAULT`, confirmé par le croisement avec `maintenance_type` |

| Manquants isolés (30) | `maintenance_provider` | inférence par site (1 seul prestataire par site) |

| Encodage des catégorielles | `maintenance_type`, `fault_code`, `maintenance_provider` | One-Hot (`df_encoded`) ; `record_status` écartée (variance nulle) |

| Valeurs extrêmes (uptime/qualité bas) | numériques | **conservées**, expliquées par le type de maintenance |



**Biais introduits et assumés :**

- l'inférence de `maintenance_provider` par site suppose qu'un site ne change jamais de prestataire sur

toute la période — vérifié sur les données disponibles, mais reste une hypothèse pour les 30 lignes concernées ;

- `record_status` étant constant (`closed`), la colonne est gardée mais n'apportera rien à l'analyse ;

- comme pour `environmental_monthly.csv`, on garde la 1ère occurrence des doublons après avoir vérifié

qu'ils étaient identiques.


# 11. Clustering (Cours 2)



### a. Choix des variables pertinentes

- **Gardées (numériques)** : `uptime_pct`, `acoustic_quality`.
- **Gardées (One-Hot)** : `maintenance_type_*`, `fault_code_*`, `maintenance_provider_*`.
- **Écartées** : identifiants (`device_operation_id`, `site_code`, `device_reference`), dates, `year` / `month`, `duration_days` (quasi constante ~28–31 j), `record_status` (une seule modalité).


In [ ]:
df_stats = pd.read_csv("cleaned/monitoring_device_operations_clean.csv")
df_encoded = pd.read_csv("cleaned/monitoring_device_operations_clean_encoded.csv")
df_stats["operation_start"] = pd.to_datetime(df_stats["operation_start"])
df_stats["operation_end"] = pd.to_datetime(df_stats["operation_end"])
df_stats["region"] = df_stats["site_code"].str[:3]

cols_cluster_num = ["uptime_pct", "acoustic_quality"]
cols_maint = [c for c in df_encoded.columns if c.startswith("maintenance_type_")]
cols_fault = [c for c in df_encoded.columns if c.startswith("fault_code_")]
cols_prov = [c for c in df_encoded.columns if c.startswith("maintenance_provider_")]
cols_cluster = cols_cluster_num + cols_maint + cols_fault + cols_prov

X = df_encoded[cols_cluster].copy()
print("Matrice clustering :", X.shape)
print("Manquants :", X.isna().sum().sum())
print("Numériques :", len(cols_cluster_num),
      "| maint :", len(cols_maint),
      "| fault :", len(cols_fault),
      "| provider :", len(cols_prov))
display(X.head())


### b. Standardisation

`StandardScaler` sur les numériques uniquement ; dummies laissées en 0/1.


In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = X.copy()
X_scaled[cols_cluster_num] = StandardScaler().fit_transform(X[cols_cluster_num])
print("Moyennes ≈ 0 :")
print(X_scaled[cols_cluster_num].mean().round(3))
print("\nÉcarts-types ≈ 1 :")
print(X_scaled[cols_cluster_num].std().round(3))


### c. K-Means : choix de k (coude + silhouette)


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

ks = range(2, 11)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ks, inertias, "o-"); ax[0].set_title("Coude (inertie)"); ax[0].set_xlabel("k")
ax[1].plot(ks, silhouettes, "o-"); ax[1].set_title("Silhouette"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.show()

print("Inerties :", dict(zip(ks, np.round(inertias, 1))))
print("Silhouettes :", dict(zip(ks, np.round(silhouettes, 3))))
best_k = int(max(ks, key=lambda k: silhouettes[k - 2]))
print("Meilleure silhouette à k =", best_k)


### - Choix de k

La silhouette est clairement meilleure à **k=2** (séparation nette normal vs panne). On retient **k=2**.


In [ ]:
k_final = 2
km = KMeans(n_clusters=k_final, n_init=10, random_state=42).fit(X_scaled)
labels_km = km.labels_
print("k retenu :", k_final)
print("Taille des clusters :", pd.Series(labels_km).value_counts().sort_index().to_dict())


### d. Classification hiérarchique (Ward) + comparaison K-Means


In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Échantillon pour le dendrogramme (affichage) ; CAH complète sur tout le jeu
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 5))
dendrogram(Z, truncate_mode="lastp", p=20, show_leaf_counts=True)
plt.title("Dendrogramme (Ward, 20 derniers regroupements)")
plt.xlabel("Taille du groupe"); plt.ylabel("Distance de fusion")
plt.tight_layout(); plt.show()

labels_hc = fcluster(Z, t=k_final, criterion="maxclust")
print("Tailles CAH :", dict(zip(*np.unique(labels_hc, return_counts=True))))
print("\nK-Means vs CAH :")
display(pd.crosstab(labels_km, labels_hc, rownames=["kmeans"], colnames=["hierarchique"]))


### e. Interprétation des clusters


In [ ]:
df_clust = df_stats.copy()
df_clust["cluster"] = labels_km

print("Taille :", df_clust["cluster"].value_counts().sort_index().to_dict())
print("\nProfil moyen (numériques) :")
display(df_clust.groupby("cluster")[cols_cluster_num + ["duration_days"]].mean().round(1))

print("\nMaintenance (%) :")
display((pd.crosstab(df_clust["cluster"], df_clust["maintenance_type"], normalize="index") * 100).round(1))
print("\nFault code (%) :")
display((pd.crosstab(df_clust["cluster"], df_clust["fault_code"], normalize="index") * 100).round(1))

print("\nFocus KNT-KES avril 2024 :")
focus = df_clust[(df_clust["site_code"] == "KNT-KES") &
                 (df_clust["operation_start"].dt.year == 2024) &
                 (df_clust["operation_start"].dt.month == 4)]
display(focus[["device_operation_id", "uptime_pct", "acoustic_quality",
               "maintenance_type", "fault_code", "cluster"]])


### - Lecture clusters

- **Cluster majoritaire (~96 %) — normal** : uptime / qualité hauts, quasi uniquement `routine` + `NO_FAULT`.
- **Petit cluster (~4 %) — panne / maintenance lourde** : uptime / qualité bas, `corrective` / `sensor replacement`, vrais `fault_code`.
- **KNT-KES avril 2024** : appartient au cluster **normal** (contrôle enquête bats).


### f. Visualisation PCA 2D


In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(X_scaled)
print("Variance expliquée :", pca.explained_variance_ratio_.round(3),
      "-> total", round(pca.explained_variance_ratio_.sum(), 3))

plt.figure(figsize=(8, 6))
sns.scatterplot(x=coords[:, 0], y=coords[:, 1], hue=labels_km, palette="Set1", alpha=0.5, s=18)
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.title(f"Clusters K-Means (k={k_final}) en PCA 2D")
plt.legend(title="cluster")
plt.tight_layout(); plt.show()


# 12. Patterns et anomalies (Cours 3)

### a. Anomalies statistiques (IQR + z-score)


In [ ]:
cols_anom = cols_cluster_num
z = (df_stats[cols_anom] - df_stats[cols_anom].mean()) / df_stats[cols_anom].std()
q1, q3 = df_stats[cols_anom].quantile(0.25), df_stats[cols_anom].quantile(0.75)
iqr = q3 - q1
masque_iqr = (df_stats[cols_anom] < q1 - 1.5 * iqr) | (df_stats[cols_anom] > q3 + 1.5 * iqr)
masque_z = z.abs() > 3

display(pd.DataFrame({"IQR": masque_iqr.sum(), "|z|>3": masque_z.sum(),
                      "les deux": (masque_iqr & masque_z).sum()}))

anom_stat = df_stats[(masque_iqr | masque_z).any(axis=1)].copy()
print("Lignes anomalies stats :", len(anom_stat), "/", len(df_stats))
print("\nMaintenance :")
display(anom_stat["maintenance_type"].value_counts())
print("\nFault code :")
display(anom_stat["fault_code"].value_counts())
print("\nTop sites :")
display(anom_stat["site_code"].value_counts().head(8))


### - Lecture anomalies stats

Les extrêmes d'uptime / qualité concentrent les interventions **non-routine** et les vrais `fault_code` → extrêmes **réels**, pas erreurs de saisie (aligné cluster panne).


### b. Anomalies multivariées (Isolation Forest + LOF)


In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

iso = IsolationForest(n_estimators=200, contamination=0.02, random_state=42).fit(X_scaled)
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.02)

res = df_stats[["device_operation_id", "site_code", "maintenance_type",
                "fault_code", "uptime_pct", "acoustic_quality"]].copy()
res["score_if"] = -iso.score_samples(X_scaled)
res["anom_if"] = iso.predict(X_scaled) == -1
res["anom_lof"] = lof.fit_predict(X_scaled) == -1
res["score_lof"] = -lof.negative_outlier_factor_
res["anom_stat"] = res.index.isin(anom_stat.index)
res["cluster"] = labels_km

print("IF :", res["anom_if"].sum(), "| LOF :", res["anom_lof"].sum(),
      "| les deux :", (res["anom_if"] & res["anom_lof"]).sum())
print("Recoupement stats : IF", (res["anom_if"] & res["anom_stat"]).sum(),
      "| LOF", (res["anom_lof"] & res["anom_stat"]).sum())

for nom in ["if", "lof"]:
    a = res[res[f"anom_{nom}"]]
    print(f"\n{nom.upper()} - maintenance :", a["maintenance_type"].value_counts().to_dict())
    print("fault :", a["fault_code"].value_counts().to_dict())

print("\nKNT-KES avril 2024 flaggé ?")
kes = res[(res["site_code"] == "KNT-KES") &
          (df_stats["operation_start"].dt.year == 2024) &
          (df_stats["operation_start"].dt.month == 4)]
display(kes)


### - Conclusion multivariée

- **IF** : retrouve surtout le profil panne / maintenance lourde (fort recoupement stats).
- **LOF** : plus souvent des cycles `routine` + `NO_FAULT` localement atypiques.
- IF ∩ LOF très faible → méthodes **complémentaires**.
- **KNT-KES avril 2024** : **non flaggé** par IF / LOF (appareil OK).


### c. Règles d'association (Apriori)

Items : région + type maintenance + fault + uptime/qualité discrétisés. Prestataire exclu (règles triviales).
`min_support=0.01` car les pannes sont rares.


In [ ]:
%pip install mlxtend -q
from mlxtend.frequent_patterns import apriori, association_rules

trans = pd.DataFrame({
    "region": df_stats["region"],
    "maint": df_stats["maintenance_type"],
    "fault": df_stats["fault_code"],
})
for col, nom in [("uptime_pct", "uptime"), ("acoustic_quality", "qualite")]:
    r = df_stats[col].rank(method="first")
    trans[nom] = pd.qcut(r, 3, labels=["bas", "moyen", "haut"]).astype(str)

basket = pd.get_dummies(trans, prefix_sep="=", dtype=bool)
freq = apriori(basket, min_support=0.01, use_colnames=True, max_len=3)
regles = association_rules(freq, metric="confidence", min_threshold=0.5)
regles = regles[regles["lift"] > 1.3].copy()
regles["si"] = regles["antecedents"].apply(lambda s: " + ".join(sorted(s)))
regles["alors"] = regles["consequents"].apply(lambda s: " + ".join(sorted(s)))

print("Itemsets :", len(freq), "| règles :", len(regles))
display(regles.sort_values("lift", ascending=False)
              .head(12)[["si", "alors", "support", "confidence", "lift"]].round(3))


### - Conclusion Apriori

Règles à lift élevé : `sensor replacement` ↔ pannes (`MICROPHONE_DRIFT`…), uptime/qualité bas ↔ interventions lourdes. Confirme EDA / clustering, **sans narratif caché** nouveau.


### d. Croiser anomalies stats/ML × qualitatives


In [ ]:
croise = res.copy()
croise["vrai_fault"] = croise["fault_code"] != "NO_FAULT"
croise["non_routine"] = croise["maintenance_type"] != "routine inspection"
croise["uptime_bas"] = croise["uptime_pct"] < 70
croise["qualitatif"] = croise["vrai_fault"] | croise["non_routine"] | croise["uptime_bas"]
croise["detectee"] = croise["anom_stat"] | croise["anom_if"] | croise["anom_lof"]

print("=== Effectifs ===")
print("Stats/IF/LOF :", croise["anom_stat"].sum(), croise["anom_if"].sum(), croise["anom_lof"].sum())
print("Qualitatif (panne déclarée / non-routine / uptime<70) :", croise["qualitatif"].sum())

print("\n=== Recoupements ===")
for col, nom in [("anom_stat", "Stats"), ("anom_if", "IF"), ("anom_lof", "LOF")]:
    inter = croise[col] & croise["qualitatif"]
    print(f"{nom} ∩ qualitatif : {inter.sum()} / {croise[col].sum()} "
          f"({100 * inter.sum() / max(croise[col].sum(), 1):.0f} %)")

suspects = croise[croise["detectee"] & ~croise["qualitatif"]]
print("\nDétectés SANS signal qualitatif :", len(suspects))
display(suspects.sort_values(["score_lof", "score_if"], ascending=False)
        .head(10)[["device_operation_id", "site_code", "maintenance_type", "fault_code",
                   "uptime_pct", "acoustic_quality", "anom_stat", "anom_if", "anom_lof"]])


### - Conclusion Bloc 3

IF / stats recoupent massivement les pannes **documentées**. Les LOF « silencieux » (routine + NO_FAULT) sont des pistes secondaires. Rôle du fichier : **fiabilité technique des capteurs**, pas une intrigue écologique cachée.


# 13. Bloc 4 : graphes et processus (Cours 4)



## Étape 1 — Définir nœuds, liens, orientation, poids

Graphe de **similarité de fiabilité** entre sites :

| Élément | Choix |
|---|---|
| **Nœuds** | 64 `site_code` |
| **Profil** | uptime moyen, qualité moyenne, taux de panne (`fault ≠ NO_FAULT`), taux non-routine |
| **Liens** | similarité cosinus des profils standardisés ≥ 0.6 |
| **Orientation** | non orienté |
| **Poids** | similarité cosinus |


In [ ]:
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

profil = df_stats.groupby("site_code").agg(
    uptime=("uptime_pct", "mean"),
    qualite=("acoustic_quality", "mean"),
    taux_panne=("fault_code", lambda s: (s != "NO_FAULT").mean()),
    taux_non_routine=("maintenance_type", lambda s: (s != "routine inspection").mean()),
)
profil["region"] = profil.index.str[:3]

cols_p = ["uptime", "qualite", "taux_panne", "taux_non_routine"]
Xp = StandardScaler().fit_transform(profil[cols_p])
S = cosine_similarity(Xp)
sites = profil.index.tolist()
SEUIL = 0.6

G = nx.Graph()
for site in sites:
    G.add_node(site, region=profil.loc[site, "region"],
               uptime=float(profil.loc[site, "uptime"]),
               taux_panne=float(profil.loc[site, "taux_panne"]))

for i in range(len(sites)):
    for j in range(i + 1, len(sites)):
        w = float(S[i, j])
        if w >= SEUIL:
            G.add_edge(sites[i], sites[j], weight=w)

print("n =", G.number_of_nodes(), "| m =", G.number_of_edges(),
      "| densité =", round(nx.density(G), 4))
print("Isolés :", list(nx.isolates(G)))
print("KNT-KES degré :", G.degree("KNT-KES") if "KNT-KES" in G else "NA")


In [ ]:
region_color = {"AVL": "#1f77b4", "KNT": "#2ca02c", "TMB": "#ff7f0e"}
colors = [region_color.get(G.nodes[n]["region"], "grey") for n in G.nodes()]
sizes = [80 + 400 * G.nodes[n]["taux_panne"] for n in G.nodes()]
pos = nx.spring_layout(G, weight="weight", seed=42, k=0.45)

plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, alpha=0.3, edge_color="grey")
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes,
                       alpha=0.9, linewidths=0.5, edgecolors="black")
labels = {n: n for n in ["KNT-KES", "KNT-GGN", "AVL-GRB", "TMB-HQL"] if n in G}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_weight="bold")
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color=c, label=r) for r, c in region_color.items()],
           title="Région", loc="upper left")
plt.title(f"Similarité fiabilité capteurs (seuil={SEUIL})\nn={G.number_of_nodes()}, m={G.number_of_edges()}")
plt.axis("off"); plt.tight_layout(); plt.show()
print("Taille nœud ∝ taux de panne du site")


## Étape 2 — Description : n, m, densité, degrés, loi de puissance


In [ ]:
from collections import Counter

n, m = G.number_of_nodes(), G.number_of_edges()
degrees = dict(G.degree())
deg_vals = np.array(list(degrees.values()))
print(f"n={n} | m={m} | densité={nx.density(G):.4f}")
print(f"degré moyen={deg_vals.mean():.2f} | min={deg_vals.min()} | max={deg_vals.max()}")
print("Top 8 degrés :", sorted(degrees.items(), key=lambda x: -x[1])[:8])
print("KNT-KES :", degrees.get("KNT-KES"))

counts = Counter(deg_vals)
ks = sorted(k for k in counts if k >= 1)
pk = [counts[k] / n for k in ks]
if len(ks) >= 3:
    b, a = np.polyfit(np.log(np.array(ks, float)), np.log(np.array(pk, float)), 1)
    print(f"Fit log-log γ ≈ {-b:.2f} (loi de puissance peu probable si γ hors 2–3)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(counts.keys()), [counts[k] for k in counts], color="steelblue", edgecolor="black")
ax.set_xlabel("Degré k"); ax.set_ylabel("Nb sites"); ax.set_title("Distribution des degrés")
plt.tight_layout(); plt.show()


## Étape 3 — Centralités + plus courts chemins


In [ ]:
composantes = sorted(nx.connected_components(G), key=len, reverse=True)
G_cc = G.subgraph(composantes[0]).copy()
for u, v, d in G_cc.edges(data=True):
    d["distance"] = 1.0 - d["weight"]

close_c = nx.closeness_centrality(G_cc, distance="distance")
betw_c = nx.betweenness_centrality(G_cc, weight="distance", normalized=True)
cent = pd.DataFrame({
    "degree": pd.Series(dict(G_cc.degree())),
    "closeness": pd.Series(close_c),
    "betweenness": pd.Series(betw_c),
    "region": pd.Series({n: G_cc.nodes[n]["region"] for n in G_cc}),
    "taux_panne": pd.Series({n: G_cc.nodes[n]["taux_panne"] for n in G_cc}),
}).sort_values("betweenness", ascending=False)

print("Composante principale :", G_cc.number_of_nodes(), "/", n)
print("\nTop 8 betweenness :")
display(cent.head(8).round(4))
print("\nSites clés :")
for site in ["KNT-KES", "KNT-GGN", "AVL-GRB", "TMB-HQL"]:
    if site in cent.index:
        r = cent.loc[site]
        print(f"{site}: deg={r['degree']:.0f} close={r['closeness']:.3f} "
              f"betw={r['betweenness']:.3f} panne={r['taux_panne']:.3f}")
    else:
        print(site, ": hors composante")


## Étape 4 — Communautés (Louvain, Girvan-Newman, Infomap, Q)


In [ ]:
# Louvain
louvain_sets = nx.community.louvain_communities(G_cc, weight="weight", seed=42)
Q_louvain = nx.community.modularity(G_cc, louvain_sets, weight="weight")
louvain_part = {n: i for i, comm in enumerate(louvain_sets) for n in comm}
print(f"Louvain : k={len(louvain_sets)}  Q={Q_louvain:.3f}")
for i, comm in enumerate(sorted(louvain_sets, key=len, reverse=True)):
    regs = pd.Series([G_cc.nodes[n]["region"] for n in comm]).value_counts().to_dict()
    flag = []
    if "KNT-KES" in comm: flag.append("KES")
    if "KNT-GGN" in comm: flag.append("GGN")
    print(f"  L{i}: n={len(comm):2d}  régions={regs}  {flag}")

# Girvan-Newman (meilleure Q entre 2 et 8)
gn_gen = nx.community.girvan_newman(G_cc)
best_Q, best_part, best_k = -1, None, None
for _ in range(2, 9):
    partition = next(gn_gen)
    Q = nx.community.modularity(G_cc, partition, weight="weight")
    if Q > best_Q:
        best_Q, best_part, best_k = Q, partition, len(partition)
print(f"\nGirvan-Newman : k={best_k}  Q={best_Q:.3f}")

# Infomap
try:
    from infomap import Infomap
    node_ids = {n: i for i, n in enumerate(G_cc.nodes())}
    id_nodes = {i: n for n, i in node_ids.items()}
    im = Infomap(silent=True, num_trials=10)
    for n_, i in node_ids.items():
        im.add_node(i, name=n_)
    for u, v, d in G_cc.edges(data=True):
        im.add_link(node_ids[u], node_ids[v], d["weight"])
    im.run()
    infomap_sets_dict = {}
    for node in im.nodes:
        infomap_sets_dict.setdefault(node.module_id, set()).add(id_nodes[node.node_id])
    infomap_sets = list(infomap_sets_dict.values())
    Q_infomap = nx.community.modularity(G_cc, infomap_sets, weight="weight")
    print(f"Infomap : k={len(infomap_sets)}  Q={Q_infomap:.3f}")
except Exception as e:
    Q_infomap = float("nan")
    print("Infomap non exécuté (bibliothèque absente ou erreur) :", e)

print(f"\nKNT-KES et KNT-GGN même communauté Louvain ? "
      f"{louvain_part.get('KNT-KES') == louvain_part.get('KNT-GGN')}")


## Étape 5 — Process mining

`monitoring_device_operations` n'est **pas** un event log de processus métier (pas de `case_id` / `activity` ordonnées au sens pm4py).  
Ce sont des **cycles mensuels d'appareils**. → Process mining **non applicable** sur ce fichier.


In [ ]:
print("Champs process-mining typiques présents ?")
for c in ["case_id", "activity", "concept:name", "time:timestamp"]:
    print(f"  {c:20s}: {'OUI' if c in df_stats.columns else 'NON'}")
print("Verdict : PAS un event log → pas de Heuristic / Inductive / Fuzzy Miner sur ce notebook.")


# 14. Bloc 5 : données spatiales et temporelles (Cours 5)



## A. Spatial

Jointure `monitoring_sites` (lat/lon). Géographie **fictive**. Avec 64 points → Moran / LISA / Voronoi **pertinents**.


In [ ]:
sites_geo = pd.read_csv("cleaned/monitoring_sites_cleaned.csv")[
    ["site_code", "region", "jurisdiction", "latitude", "longitude"]
]
profil_geo = profil.reset_index().merge(sites_geo, on="site_code", how="left")
print("Sites avec GPS :", profil_geo[["latitude", "longitude"]].notna().all(axis=1).sum(), "/", len(profil_geo))

fig, ax = plt.subplots(figsize=(11, 7))
colors = profil_geo["region_code"] if "region_code" in profil_geo else profil_geo["site_code"].str[:3]
profil_geo = profil_geo.copy()
profil_geo["region_code"] = profil_geo["site_code"].str[:3]
c = profil_geo["region_code"].map(region_color)
sizes = 40 + 800 * profil_geo["taux_panne"]
ax.scatter(profil_geo["longitude"], profil_geo["latitude"], c=c, s=sizes,
           alpha=0.85, edgecolors="black", linewidths=0.5)
for _, r in profil_geo[profil_geo["site_code"].isin(["KNT-KES", "KNT-GGN"])].iterrows():
    ax.annotate(r["site_code"], (r["longitude"], r["latitude"]),
                textcoords="offset points", xytext=(5, 5), fontsize=8, fontweight="bold")
ax.legend(handles=[Patch(color=col, label=reg) for reg, col in region_color.items()],
          title="Région")
ax.set_title("Carte sites — taille ∝ taux de panne capteur")
ax.set_xlabel("Longitude (fictive)"); ax.set_ylabel("Latitude (fictive)")
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


In [ ]:
%pip install libpysal esda -q
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
from scipy.spatial import Voronoi, voronoi_plot_2d

var = "taux_panne"
coords = profil_geo[["longitude", "latitude"]].values
w = KNN.from_array(coords, k=4)
w.transform = "R"
y = profil_geo[var].values
mi = Moran(y, w)
print(f"Moran I GLOBAL sur `{var}` : I={mi.I:.3f}  p={mi.p_sim:.4f}")
print("(peu informatif si 3 régions géo-disjointes — Moran par région :)")

for reg in ["AVL", "KNT", "TMB"]:
    sub = profil_geo[profil_geo["region_code"] == reg]
    w_r = KNN.from_array(sub[["longitude", "latitude"]].values, k=min(3, len(sub) - 1))
    w_r.transform = "R"
    mi_r = Moran(sub[var].values, w_r)
    print(f"  {reg}: I={mi_r.I:.3f}  p={mi_r.p_sim:.4f}")

lisa = Moran_Local(y, w)
labels_q = {1: "High-High", 2: "Low-High", 3: "Low-Low", 4: "High-Low"}
profil_geo = profil_geo.copy()
profil_geo["lisa_q"] = lisa.q
profil_geo["lisa_sig"] = lisa.p_sim < 0.05
print("\nLISA significatifs :", int(profil_geo["lisa_sig"].sum()))
for site in ["KNT-KES", "KNT-GGN"]:
    row = profil_geo[profil_geo["site_code"] == site].iloc[0]
    print(f"{site}: taux_panne={row[var]:.3f}  Q={labels_q[row['lisa_q']]}  sig={row['lisa_sig']}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
q_colors = {1: "#d7191c", 2: "#abd9e9", 3: "#2c7bb6", 4: "#fdae61"}
c_lisa = [q_colors[q] if sig else "#cccccc"
          for q, sig in zip(profil_geo["lisa_q"], profil_geo["lisa_sig"])]
axes[0].scatter(profil_geo["longitude"], profil_geo["latitude"], c=c_lisa, s=80, edgecolors="black", linewidths=0.5)
axes[0].set_title("LISA taux de panne (gris = non sig.)")
vor = Voronoi(coords)
voronoi_plot_2d(vor, ax=axes[1], show_vertices=False, line_colors="grey", line_width=0.8, point_size=0)
axes[1].scatter(profil_geo["longitude"], profil_geo["latitude"],
                c=profil_geo["region_code"].map(region_color), s=50, edgecolors="black", linewidths=0.4)
axes[1].set_title("Voronoi des 64 sites")
axes[1].set_xlim(profil_geo["longitude"].min() - 1, profil_geo["longitude"].max() + 1)
axes[1].set_ylim(profil_geo["latitude"].min() - 1, profil_geo["latitude"].max() + 1)
plt.tight_layout(); plt.show()


## B. Temporel

Séries mensuelles d'uptime / qualité (global + KNT-KES). STL + ACF + anomalies fenêtre glissante.


In [ ]:
series_global = df_stats.groupby("operation_start")[["uptime_pct", "acoustic_quality"]].mean().sort_index()
series_kes = (df_stats[df_stats["site_code"] == "KNT-KES"]
              .set_index("operation_start")[["uptime_pct", "acoustic_quality"]].sort_index())

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for ax, col in zip(axes, ["uptime_pct", "acoustic_quality"]):
    ax.plot(series_global.index, series_global[col], color="steelblue", lw=1.4, label="global")
    ax.plot(series_kes.index, series_kes[col], color="darkgreen", lw=1.2, alpha=0.85, label="KNT-KES")
    ax.axvline(pd.Timestamp("2024-04-01"), color="red", ls="--", lw=1, label="avr. 2024")
    ax.set_title(col); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
fig.suptitle("Séries mensuelles — uptime / qualité", fontweight="bold")
plt.tight_layout(); plt.show()
print("Global :", len(series_global), "mois | KNT-KES :", len(series_kes), "mois")


In [ ]:
%pip install statsmodels -q
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf

y = series_global["uptime_pct"].asfreq("MS")
stl = STL(y, period=12, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(13, 8), sharex=True)
axes[0].plot(stl.observed); axes[0].set_ylabel("Observé"); axes[0].set_title("STL — uptime_pct (global)")
axes[1].plot(stl.trend); axes[1].set_ylabel("Tendance")
axes[2].plot(stl.seasonal); axes[2].set_ylabel("Saisonnalité")
axes[3].plot(stl.resid); axes[3].set_ylabel("Résidu")
axes[3].axvline(pd.Timestamp("2024-04-01"), color="red", ls="--", lw=1)
for ax in axes: ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
plot_acf(y.dropna(), lags=24, ax=ax, title="ACF — uptime global")
plt.tight_layout(); plt.show()


In [ ]:
def rolling_z_anomalies(s, window=12, thresh=3.0):
    mu = s.rolling(window, center=True, min_periods=6).mean()
    sd = s.rolling(window, center=True, min_periods=6).std()
    z = (s - mu) / sd
    return z, z.abs() > thresh

avril = pd.Timestamp("2024-04-01")
print("=== KNT-KES avril 2024 — z glissant ===")
for col in ["uptime_pct", "acoustic_quality"]:
    z, mask = rolling_z_anomalies(series_kes[col])
    zv = z.loc[avril] if avril in z.index else np.nan
    print(f"  {col:20s}  z={zv:6.2f}  anomalie={bool(mask.loc[avril]) if avril in mask.index else 'NA'}")

print("\n=== Focus ligne brute KNT-KES avril 2024 ===")
display(df_stats[(df_stats["site_code"] == "KNT-KES") &
                 (df_stats["operation_start"] == avril)]
        [["device_operation_id", "uptime_pct", "acoustic_quality",
          "maintenance_type", "fault_code", "maintenance_provider"]])


### - Lecture Bloc 5

- Carte : 3 blocs régionaux ; taille ∝ taux de panne.
- Moran : structure spatiale des pannes souvent **faible** (pannes techniques dispersées).
- Temporel : cycles stables ; **KNT-KES avril 2024** = `routine` + `NO_FAULT`, z non extrême → **capteur OK** pendant le crash bats.


# 15. Bloc 6 : données textuelles (Cours 6)

`monitoring_device_operations` n'a **pas** de champ narratif (`text` / `title` / mémo).  
Uniquement des codes courts (`fault_code`, `maintenance_type`). → Bloc 6 **non applicable** (documenté pour le rapport).


In [ ]:
cols = df_stats.columns.tolist()
text_like = [c for c in cols if c.lower() in
             {"text", "title", "body", "memo", "content", "description", "summary", "notes"}]
print("Colonnes narratives :", text_like if text_like else "(aucune)")
print("Verdict : PAS un corpus textuel → Bloc 6 non appliqué.")


## SYNTHÈSE FINALE — MONITORING_DEVICE_OPERATIONS


In [ ]:
print("\n" + "=" * 80)
print("SYNTHÈSE — MONITORING_DEVICE_OPERATIONS (fiabilité des capteurs)")
print("=" * 80 + "\n")

summary = {
    "Relevés (nettoyés)": f"{len(df_stats):,}".replace(",", " "),
    "Sites / appareils": f"{df_stats['site_code'].nunique()} (1 appareil / site)",
    "Période": f"{df_stats['operation_start'].min().date()} → {df_stats['operation_start'].max().date()}",
    "Clustering (Bloc 2)": "k=2 — normal (~96%) vs panne/maintenance lourde (~4%)",
    "Anomalies (Bloc 3)": "IF ≈ profil panne ; LOF ≈ atypies locales routine",
    "Graphe (Bloc 4)": "Similarité fiabilité entre sites ; process mining N/A",
    "Spatial/temporel (5)": "64 GPS ; KNT-KES avr. 2024 = routine + NO_FAULT",
    "Texte (Bloc 6)": "Non applicable",
    "Rôle dossier": "Contrôle technique : panne capteur ≠ cause du crash bats",
    "Verdict KNT-KES avr.2024": "Appareil OK → cause bats ailleurs (EFR-510088)",
}
for k, v in summary.items():
    print(f"  {k:.<42} {v}")


BAT-0000064 (bat_surveys)

    ↓ ANOMALIE NUMÉRIQUE

    Chute de -65% du nombre de chauve-souris
    (site KNT-KES, avril 2024)

         ↕ LIEN DIRECT

EFR-510088 (environmental_field_reports)

    ↓ ANOMALIE TEXTUELLE / MATÉRIELLE

    EVIDENCE: filets coupés + traces d'intrusion
    CAUSALITÉ: Filets → Braconnage → Perte d'individus

         ↕ CONTRÔLE TECHNIQUE (ce fichier)

monitoring_device_operations (KNT-KES, avril 2024)

    ↓ FIABILITÉ CAPTEUR

    maintenance = routine inspection
    fault_code = NO_FAULT
    uptime / qualité dans le cluster NORMAL
    pas d'anomalie IF / LOF / z glissant extrême

    VERDICT : panne d'appareil ÉCARTÉE
              → renforce bat ↔ field reports
                 (braconnage, pas un capteur en panne)


# Synthèse des résultats

## 1. De quoi parle ce fichier ?

Ce fichier suit les **appareils de monitoring** (micros / capteurs) sur les sites.

Une ligne = **un mois** pour un appareil :
- est-ce qu'il a bien tourné ? (`uptime_pct`)
- la qualité du son est-elle bonne ? (`acoustic_quality`)
- a-t-on fait une maintenance ?
- y avait-il une panne ? (`fault_code`)

---

## 2. Résultat principal

On voit **2 groupes** très clairs :

### Groupe A — Tout va bien (~96 %)
- appareil qui marche
- bonne qualité
- maintenance de routine
- **pas de panne** (`NO_FAULT`)

### Groupe B — L'appareil dysfonctionne (~4 %)
- uptime bas
- qualité basse
- vraie réparation ou remplacement de capteur
- vrai code de panne (ex. micro dérivé, mémoire pleine, coupure de courant)

Donc les "anomalies" ici, ce n'est **pas un mystère** :  
ce sont juste les moments où **la machine est en panne**, et le fichier le dit déjà.

---

## 3. Le point important pour l'enquête chauves-souris

L'enquête ecology part de ça :

> À **KNT-KES**, en **avril 2024**, la colonie chute d'environ **−65 %**  
> (relevé `BAT-0000064` : 27,9 en avril 2023 → 9,5 en avril 2024)

Question : et si c'était juste un **capteur cassé** qui a mal mesuré ?

### Réponse de ce fichier (KNT-KES, avril 2024)
- maintenance = **routine**
- panne = **NO_FAULT**
- groupe = **normal** (pas le groupe panne)

**Conclusion :** le capteur **marchait**.  
Ce n'est **pas** une panne d'appareil qui explique la chute des chauves-souris.

---

## 4. Verdict final (en une phrase)

Ce fichier sert de **contrôle technique** :  
il écarte la piste "capteur en panne" et renforce l'idée que la cause est ailleurs  
(→ rapport terrain `EFR-510088` : filets coupés / intrusion / braconnage).
